# Parte 1 – Visualización con Plotly

Usamos **Plotly Express**, como en la sesión 6: gráficos interactivos con una línea de código.

In [1]:
import pandas as pd
import plotly.express as px

## 1. Cargar el dataset

In [2]:
# El ubigeo se lee como texto para no perder el cero de la izquierda ("01", no 1)
datos = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})
print("Dimensiones:", datos.shape)
datos.head()

Dimensiones: (25, 9)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
1,02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
2,03,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,1,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,4,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,5,1


## 2. Tema y columnas

**Pregunta:** *¿El Estado declara emergencia donde más llueve?* Usamos nuestra `tabla_final.csv`
del Assignment 2: una fila por departamento, temporada 1 de enero – 31 de mayo de 2023.

Columnas que usaremos:

- `departamento`: nombre del departamento (25, incluido el Callao).
- `lluvia_total_mm`: lluvia acumulada en la capital del departamento (Open-Meteo).
- `dias_lluvia_fuerte`: días con 20 mm o más.
- `declaratorias`: decretos supremos de la PCM que declaran emergencia por lluvias en el departamento.

## 3. Gráfico de dispersión: lluvia vs. declaratorias

Cada punto es un departamento; al pasar el mouse aparece su nombre (`hover_name`).
El tamaño del punto son los días de lluvia fuerte.

In [3]:
fig_dispersion = px.scatter(
    datos, x="lluvia_total_mm", y="declaratorias",
    size="dias_lluvia_fuerte", hover_name="departamento",
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "declaratorias": "Declaratorias de emergencia",
            "dias_lluvia_fuerte": "Días ≥ 20 mm"},
    title="Más lluvia no significa más declaratorias de emergencia",
)
fig_dispersion.update_layout(
    template="plotly_white",
    annotations=[dict(
        text="Fuente: decretos supremos de la PCM (gob.pe) y Open-Meteo, ene–may 2023. Elaboración propia.",
        showarrow=False, xref="paper", yref="paper",
        x=0, y=-0.18, font=dict(size=11, color="gray"),
    )],
)
fig_dispersion.show()

In [4]:
correlacion = datos["lluvia_total_mm"].corr(datos["declaratorias"])
print("Correlación entre lluvia y declaratorias:", round(correlacion, 2))

Correlación entre lluvia y declaratorias: -0.24


La correlación es negativa: los departamentos con más lluvia tienden a tener **menos** declaratorias.

## 4. Tercer gráfico: histograma de la lluvia

Muestra cómo se reparte la lluvia entre departamentos.

In [5]:
fig_histograma = px.histogram(
    datos, x="lluvia_total_mm", nbins=8,
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)"},
    title="La mayoría de capitales recibió menos de 900 mm; solo cinco pasaron de 1,000",
)
fig_histograma.update_layout(
    template="plotly_white",
    yaxis_title="Número de departamentos",
    annotations=[dict(
        text="Fuente: decretos supremos de la PCM (gob.pe) y Open-Meteo, ene–may 2023. Elaboración propia.",
        showarrow=False, xref="paper", yref="paper",
        x=0, y=-0.18, font=dict(size=11, color="gray"),
    )],
)
fig_histograma.show()

## 5. Verificación rápida: los extremos de la tabla coinciden con los gráficos

Revisamos en la tabla los valores más altos y más bajos y los comparamos con lo que muestran los gráficos.

In [6]:
print("Más declaratorias:")
print(datos.nlargest(3, "declaratorias")[["departamento", "declaratorias", "lluvia_total_mm"]])
print("\nMenos declaratorias:")
print(datos.nsmallest(3, "declaratorias")[["departamento", "declaratorias", "lluvia_total_mm"]])

Más declaratorias:
   departamento  declaratorias  lluvia_total_mm
14         Lima              7             55.5
1        Áncash              6           1145.8
4      Ayacucho              5            498.4

Menos declaratorias:
     departamento  declaratorias  lluvia_total_mm
16  Madre de Dios              0           1248.4
2        Apurímac              1            726.2
6          Callao              1             55.5


In [7]:
print("Más lluvia:")
print(datos.nlargest(5, "lluvia_total_mm")[["departamento", "lluvia_total_mm"]])
print("\nMenos lluvia:")
print(datos.nsmallest(3, "lluvia_total_mm")[["departamento", "lluvia_total_mm"]])
print("\nCapitales con más de 1,000 mm:", (datos["lluvia_total_mm"] > 1000).sum())

Más lluvia:
     departamento  lluvia_total_mm
15         Loreto           1604.6
24        Ucayali           1316.0
16  Madre de Dios           1248.4
1          Áncash           1145.8
21     San Martín           1050.5

Menos lluvia:
   departamento  lluvia_total_mm
22        Tacna             28.2
6        Callao             55.5
14         Lima             55.5

Capitales con más de 1,000 mm: 5


Coincide con los gráficos:

- **Dispersión:** el punto más a la derecha es Loreto (1,605 mm) con solo 1 declaratoria, y el punto más alto es Lima (7), pegado a la izquierda (55 mm).
- **Histograma:** las cinco capitales con más de 1,000 mm forman las barras de la derecha, y Tacna (28 mm) es la de menos lluvia.